In [1]:
import sqlite3
import re
import itertools
import string
import time

In [169]:
conn = sqlite3.connect("../Metal_Forge.db")

def regexp(pattern, item):
    if item is None:
        return False
    # Returns True if the regex pattern is found in the item
    return re.search(pattern, item) is not None

conn.create_function("REGEXP", 2, regexp)

cur = conn.cursor()

cur.execute("SELECT pattern FROM patterns")

patterns = [re.compile(pat[0]) for pat in cur.fetchall()]

cur.execute("SELECT morph FROM morphs")

morphs = [morph[0] for morph in cur.fetchall()]

In [93]:
import numpy as np

class Morph:
    # class attributes
    # x = 0
    patterns_global = patterns
    
    def __init__(self, morph: str):
        # instance attributes
        self.morph = morph
        self.simple_morph = self.simplify_morph(morph)
        self.trigram_decomp = self.decomp_trigrams(self.simple_morph)
        self.patterns = [p for p in self.patterns_global if bool(re.search(p, morph))]
        self.trigram_decomp_set = set(self.trigram_decomp)
        self.patterns_set = set(self.patterns)
    
    def simplify_morph(self, morph):
        char_map = {"ą": "a", "č": "c", "ę": "e", "ė": "e", "į": "i", "š": "s", "ų": "u", "ū": "u", "ž": "z"}
        simple_morph = "".join(char_map.get(c, c) for c in morph)
        return simple_morph
    
    def decomp_trigrams(self, morph):
        morph_trigrams = []
        for i in range(len(morph) - 2):
            morph_trigrams.append(morph[i:i+3])
        return morph_trigrams


class Cluster:
    def __init__(self, cluster: list):
        self.cluster = cluster
        self.rating_total = self.calc_rating_total()
        self.morph_supports = self.calc_morph_supports()


    def calc_rating_total(self):
        s = 0
        # Iterate through unique pairs
        for i, m1 in enumerate(self.cluster):
            for m2 in self.cluster[i+1:]:
                s += self.get_pair_rating(m1, m2)
        return round(s, 2)
    
    def calc_morph_supports(self):
        supp_arr = np.array([])
        for i, m1 in enumerate(self.cluster):
            s = 0
            for j, m2 in enumerate(self.cluster[0:i] + self.cluster[i+1:]):
                s += self.get_pair_rating(m1, m2)
            supp_arr = np.append(supp_arr, s)
        return supp_arr

    @staticmethod 
    def get_pair_rating(morph_1, morph_2):
        intersection = len(morph_1.trigram_decomp_set & morph_2.trigram_decomp_set)
        union = len(morph_1.trigram_decomp_set | morph_2.trigram_decomp_set)
        return round(intersection / union, 2)
    
    def calc_replacement_supports(self, outside_morph: Morph):
        supp_arr = np.array([])
        for i in range(len(self.cluster)):
            s = 0
            for j, m in enumerate(self.cluster[0:i]+self.cluster[i+1:]):
                s += self.get_pair_rating(outside_morph, m)
            supp_arr = np.append(supp_arr, s)
        return supp_arr
    
    def replace_member(self, index, new_morph):
        self.cluster[index] = new_morph
        self.rating_total = self.calc_rating_total()
        self.morph_supports = self.calc_morph_supports()
        return None

    def remove_member(self, index):
        removed_morph = self.cluster.pop(index)
        self.rating_total = self.calc_rating_total()
        self.morph_supports = self.calc_morph_supports()
        return removed_morph

    def sync(self):
        self.rating_total = self.calc_rating_total()
        self.morph_supports = self.calc_morph_supports()
        return None

In [98]:
import random
from pprint import pprint
clusters = []
unused_morphs = morphs

def cluster_unused_morphs(clusters, unused_morphs):
    index = random.randrange(len(unused_morphs))
    current_morph = Morph(unused_morphs.pop(index)) 

    while unused_morphs:
        appended = False
        best_cl_idx = None
        best_member_idx = None
        best_delta = 0.0001
        
        for i, cl in enumerate(clusters):
            if len(cl.cluster) < 7 and best_delta <= 0.0001:
                cl.cluster.append(current_morph)
                cl.sync()
                appended = True
                break
            
            cl_deltas = cl.calc_replacement_supports(current_morph) - cl.morph_supports
            idx = np.argmax(cl_deltas)
            
            if cl_deltas[idx] > best_delta:
                best_delta = cl_deltas[idx]
                best_cl_idx = i
                best_member_idx = idx

        if appended:
            new_idx = random.randrange(len(unused_morphs))
            current_morph = Morph(unused_morphs.pop(new_idx))

            while len(current_morph.morph) < 3:
                new_idx = random.randrange(len(unused_morphs))
                current_morph = Morph(unused_morphs.pop(new_idx))

        
        elif best_cl_idx is not None:
            target_cl = clusters[best_cl_idx]
            kicked_out_morph = target_cl.cluster[best_member_idx]
            target_cl.replace_member(best_member_idx, current_morph)
            current_morph = kicked_out_morph

        else:
            clusters.append(Cluster([current_morph]))
            new_idx = random.randrange(len(unused_morphs))
            current_morph = Morph(unused_morphs.pop(new_idx))
            
            while len(current_morph.morph) < 3:
                new_idx = random.randrange(len(unused_morphs))
                current_morph = Morph(unused_morphs.pop(new_idx))
    
    return clusters

clusters = cluster_unused_morphs(clusters=clusters, unused_morphs=unused_morphs)

In [ ]:
import pickle

# with open("saved_clusters.pkl", "wb") as file:
#     pickle.dump(clusters, file)

with open("saved_clusters.pkl", "rb") as file:
    clusters = pickle.load(file)

In [111]:
initiate_iteration = True

for k in range(8000):
    best_cl_idx = None
    best_member_idx = None
    best_delta = 0.0001
    appended = False

    if initiate_iteration:
        curr_cluster = random.choice(clusters)
        curr_morph_idx = random.randrange(len(curr_cluster.cluster))
        current_morph_support = curr_cluster.morph_supports[curr_morph_idx]
        current_morph = curr_cluster.remove_member(curr_morph_idx)
        

        for i, cl in enumerate(clusters):
            
            if curr_cluster == cl: continue

            cl_replacement_supports = cl.calc_replacement_supports(current_morph)
            cl_deltas = cl_replacement_supports - cl.morph_supports
            idx = np.argmax(cl_deltas)
            
            if cl_deltas[idx] > best_delta and cl_replacement_supports[idx] > current_morph_support:
                best_delta = cl_deltas[idx]
                best_cl_idx = i
                best_member_idx = idx

    else:
        clusters = sorted(clusters, key=lambda c: len(c.cluster), reverse=True)
        for i, cl in enumerate(clusters):

            if len(cl.cluster) < 7 and best_delta <= 0.0001:
                cl.cluster.append(current_morph)
                cl.sync()
                appended = True
                break
            
            cl_replacement_supports = cl.calc_replacement_supports(current_morph)
            cl_deltas = cl_replacement_supports - cl.morph_supports
            idx = np.argmax(cl_deltas)
            
            if cl_deltas[idx] > best_delta:
                best_delta = cl_deltas[idx]
                best_cl_idx = i
                best_member_idx = idx

    if best_cl_idx is not None:
        target_cl = clusters[best_cl_idx]
        kicked_out_morph = target_cl.cluster[best_member_idx]
        target_cl.replace_member(best_member_idx, current_morph)
        current_morph = kicked_out_morph
        initiate_iteration = False
    elif initiate_iteration:
        curr_cluster.cluster.append(current_morph)
        curr_cluster.sync()
        initiate_iteration = True
    elif appended:
        initiate_iteration = True
        continue
    else:
        clusters.append(Cluster([current_morph]))
        initiate_iteration = True
    
    clusters = [cl for cl in clusters if len(cl.cluster) > 0]


In [77]:
current_morph.morph

'giedras'

In [78]:
[m.morph for m in clusters[curr_cluster_idx].cluster]

['priedurniai', 'rieda', 'giedras', 'riedulys', 'riedu', 'gieda', 'priedu']

In [ ]:
clusters[curr_cluster_idx].morph_supports

array([1.39, 2.03, 0.91, 1.69, 2.42, 1.41, 2.31])

In [80]:
[m.morph for m in clusters[best_cl_idx].cluster]

['drakula', 'kūdra', 'kadrai', 'drausmė', 'draugas', 'viedrą', 'audra']

In [ ]:
if all(len(m) < 3 for m in unused_morphs):
    new_cluster_idx = random.randrange(len(clusters))
    new_idx = random.randrange(len(clusters[new_cluster_idx]))
    current_morph = clusters[new_cluster_idx].cluster[new_idx]



In [27]:
len(unused_morphs)

0

In [28]:
len(morphs)

0

In [145]:
[set(m.trigram_decomp) for m in arr]

AttributeError: 'list' object has no attribute 'trigram_decomp'

In [147]:
arr = [[m for m in cl.cluster] for cl in clusters if "baletas" in [m.morph for m in cl.cluster]][0]
arr

set.intersection(*[set(m.trigram_decomp) for m in arr])


{'eta', 'let', 'tas'}

In [119]:
pprint([[m.morph for m in cl.cluster] for cl in clusters if "baletas" in [m.morph for m in cl.cluster]], width=200, compact=False)

[['galėtas', 'galetas', 'arbaletas', 'valetas', 'baletas', 'skeletas', 'tualetas']]


In [115]:
print([cl.rating_total for cl in clusters])


[10.51, 9.6, 14.5, 8.63, 12.8, 7.86, 11.66, 15.06, 12.5, 10.15, 9.17, 9.16, 12.02, 9.8, 12.8, 9.8, 12.03, 10.69, 15.39, 11.63, 13.2, 9.08, 9.62, 7.74, 11.52, 8.94, 9.14, 9.02, 5.92, 12.16, 6.63, 10.32, 12.83, 6.96, 8.52, 5.6, 7.37, 9.24, 12.3, 10.62, 11.34, 9.6, 9.97, 10.41, 8.49, 15.25, 11.5, 9.33, 8.6, 9.06, 9.55, 10.09, 9.1, 16.03, 10.84, 9.24, 7.67, 8.34, 7.17, 6.14, 13.97, 7.88, 7.15, 9.11, 10.31, 8.1, 6.95, 9.36, 10.5, 9.35, 8.79, 10.33, 10.9, 8.59, 8.68, 7.68, 12.47, 7.01, 8.3, 8.12, 10.91, 8.45, 12.58, 12.3, 6.04, 7.82, 8.46, 10.87, 11.2, 10.8, 11.46, 11.0, 5.46, 12.72, 11.5, 8.62, 10.36, 8.33, 6.64, 12.0, 6.95, 6.19, 9.66, 8.17, 9.89, 4.95, 8.87, 10.71, 10.25, 11.06, 7.14, 10.66, 9.23, 7.85, 8.54, 6.78, 10.14, 7.42, 11.56, 9.84, 5.05, 10.26, 7.25, 9.75, 13.45, 9.33, 8.71, 13.75, 9.67, 13.42, 5.58, 13.25, 11.05, 12.0, 12.3, 8.39, 11.1, 11.35, 9.24, 9.42, 7.68, 10.25, 11.76, 7.57, 7.33, 6.64, 6.66, 10.56, 9.33, 8.07, 9.37, 7.26, 9.98, 10.56, 5.86, 10.09, 14.17, 9.42, 8.53, 10.47

In [ ]:
dud = dict()

dud["aus,baus,bus,medaus"] = "a"

target_set = {"baus", "medaus", "aus", "bus"}

list(target_set)

['bus', 'aus', 'baus', 'medaus']

In [184]:
morphs_classed = [Morph(m) for m in morphs]

In [186]:
trigram_dict[target_set]

{}

In [202]:
trigram_dict = dict()
# k = Morph("ankštis")
for i, m in enumerate(morphs_classed):
    for k in morphs_classed[i+1:]:
        if len(m.trigram_decomp_set & k.trigram_decomp_set) > 2:
            target_set = frozenset(m.trigram_decomp_set & k.trigram_decomp_set)
            if target_set not in trigram_dict:
                trigram_dict[target_set] = set()
            trigram_dict[target_set].add(m.morph)
            trigram_dict[target_set].add(k.morph)

trigram_dict

{frozenset({'alo', 'lot', 'oto', 'sal', 'tos'}): {'bulviųsalotos',
  'salotos',
  'salotosikras'},
 frozenset({'alo', 'lot', 'sal'}): {'bulviųsalotos',
  'salota',
  'salotos',
  'salotosikras'},
 frozenset({'bul', 'lvi', 'ulv'}): {'bulviakasis', 'bulviųsalotos'},
 frozenset({'kst', 'nks', 'sti'}): {'ankšties',
  'ankštis',
  'inkšti',
  'minkštimas'},
 frozenset({'kst', 'sti', 'tis'}): {'ankštis',
  'aukštis',
  'daugiaaukštis',
  'driežiūkštis',
  'makštis',
  'paukštis',
  'penkiaaukštis',
  'pernykštis',
  'rakštis',
  'švirkštis'},
 frozenset({'ank', 'kst', 'nks'}): {'ankšties', 'ankštis', 'lankstyti'},
 frozenset({'ank', 'nks', 'sti', 'tis'}): {'ankštis', 'kojųrankšluostis'},
 frozenset({'ank', 'kst', 'nks', 'sti'}): {'ankšties', 'ankštis'},
 frozenset({'aiv', 'iva', 'lai', 'vas'}): {'blaivas',
  'laivas',
  'laivasolandų',
  'olandųlaivas',
  'tarslaivas'},
 frozenset({'lai', 'rsl', 'sla'}): {'purslai', 'tarslaivas', 'verslai'},
 frozenset({'aly', 'lyt', 'val'}): {'pavalyt', 'va

In [ ]:
{1,2,3,4}.add(x for x in [1,2,3,4,5])

In [211]:
kos = {m for c in trigram_dict.values() for m in c if "gaidys" in c}
# atos = { l for k, l in trigram_dict.items() if "prostogąprąlįsti" in l}
# atos
kos

{'baidys', 'gaidys', 'laidys'}